# NOTEBOOK 0: CHUẨN BỊ DỮ LIỆU VQA TỪ HUGGING FACE
**Sinh viên thực hiện:** Lê Khắc Thanh Trí - **MSSV:** 523H0186
**Chiến lược:** Lấy mẫu phân tầng 30 class món ăn, mỗi class 70 ảnh (Tổng 2100 ảnh). Dùng Gemini 2.5 Flash sinh Q&A.

In [ ]:
!pip install -q -U datasets google-generativeai

import os
import time
import json
import pandas as pd
from datasets import load_dataset
import google.generativeai as genai
from google.colab import userdata, drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# Khai báo đường dẫn
BASE_DIR = '/content/drive/MyDrive/VQA_MonAnVietNam'
DATA_DIR = os.path.join(BASE_DIR, 'data')
IMG_DIR = os.path.join(DATA_DIR, 'images')
CSV_PATH = os.path.join(DATA_DIR, 'vqa_dataset.csv')

os.makedirs(IMG_DIR, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Mounted at /content/drive


In [ ]:
print("Đang tải dataset từ Hugging Face...")
hf_dataset = load_dataset("TuyenTrungLe/vietnamese_food_images", split="train")

# Lấy danh sách tên các món ăn (class labels)
labels = hf_dataset.features['label'].names
target_classes = labels[:30] # Chỉ lấy 30 món đầu tiên

# Biến đếm số lượng ảnh mỗi món
class_counts = {cls: 0 for cls in target_classes}
saved_image_paths = []

print("Đang trích xuất và lưu 2100 ảnh xuống Drive (lần đầu chạy sẽ mất vài phút)...")

for item in hf_dataset:
    label_id = item['label']
    label_name = labels[label_id]

    # Nếu thuộc 30 món mục tiêu và chưa đủ 70 ảnh
    if label_name in target_classes and class_counts[label_name] < 70:
        # Tạo thư mục cho món ăn này trên Drive
        class_dir = os.path.join(IMG_DIR, label_name)
        os.makedirs(class_dir, exist_ok=True)

        # Đường dẫn lưu ảnh: ví dụ images/pho/pho_001.jpg
        img_filename = f"{label_name}_{class_counts[label_name]:03d}.jpg"
        img_save_path = os.path.join(class_dir, img_filename)
        rel_path = f"images/{label_name}/{img_filename}"

        # Nếu ảnh chưa tồn tại trên Drive thì mới lưu lại (tránh lưu đè tốn thời gian)
        if not os.path.exists(img_save_path):
            item['image'].convert('RGB').save(img_save_path)

        saved_image_paths.append({
            'abs_path': img_save_path,
            'rel_path': rel_path,
            'label': label_name
        })

        class_counts[label_name] += 1

    # Dừng vòng lặp nếu đã gom đủ 30 * 70 = 2100 ảnh
    if len(saved_image_paths) == 30 * 70:
        break

print(f"Hoàn tất! Đã chuẩn bị sẵn sàng {len(saved_image_paths)} ảnh vật lý trên Drive.")

Đang tải dataset từ Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00006.parquet:   0%|          | 0.00/491M [00:00<?, ?B/s]

data/train-00001-of-00006.parquet:   0%|          | 0.00/544M [00:00<?, ?B/s]

...

data/test-00001-of-00002.parquet:   0%|          | 0.00/440M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17581 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2515 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5040 [00:00<?, ? examples/s]

Đang trích xuất và lưu 2100 ảnh xuống Drive (lần đầu chạy sẽ mất vài phút)...
Hoàn tất! Đã chuẩn bị sẵn sàng 2100 ảnh vật lý trên Drive.


In [ ]:
from google.colab import userdata
# Thiết lập API Key
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-2.5-flash')

def generate_vqa_for_image(image_path, label_name, max_retries=3):
    from PIL import Image
    import time
    import os
    import json

    for attempt in range(max_retries):
        try:
            img = Image.open(image_path)

            prompt = f"""
            Đây là hình ảnh món: {label_name}.
            Đóng vai chuyên gia AI, tạo 3 cặp câu hỏi và trả lời tiếng Việt liên quan đến ảnh này.
            Yêu cầu:
            1. Câu hỏi đa dạng: yes/no, thành phần, màu sắc, nhận dạng, thuộc tính, không gian.
            2. Câu trả lời cực kỳ ngắn gọn (tối đa 10 từ).
            3. CHỈ trả về mảng JSON chuẩn, KHÔNG CÓ text thừa hay markdown.
            Ví dụ:
            [
                {{"question": "Đây là món ăn gì?", "answer": "{label_name.replace('_', ' ')}."}},
                {{"question": "Món ăn này có nước dùng không?", "answer": "Có."}}
            ]
            """
            response = model.generate_content([prompt, img])
            clean_text = response.text.strip().replace('```json', '').replace('```', '')
            return json.loads(clean_text)

        except Exception as e:
            error_str = str(e)
            # Bắt đúng bệnh 429 (Vượt quá giới hạn phút)
            if '429' in error_str or 'Quota' in error_str:
                print(f"\n[Cảnh báo] Vượt tốc độ (Lỗi 429). Hệ thống tự động nghỉ 60 giây để hồi phục API... (Lần {attempt+1}/{max_retries})")
                time.sleep(60)
            # Bắt bệnh 503 (Server quá tải)
            elif '503' in error_str:
                print(f"\n[Cảnh báo] Server Google bận (Lỗi 503). Đang đợi 15 giây... (Lần {attempt+1}/{max_retries})")
                time.sleep(15)
            else:
                print(f"\n[Lỗi] Bỏ qua ảnh {os.path.basename(image_path)} vì lỗi lạ: {error_str}")
                return []

    print(f"\n[Thất bại] Đã thử {max_retries} lần nhưng vẫn lỗi. Bỏ qua ảnh này.")
    return []

In [ ]:
from tqdm.auto import tqdm
import json

# Kiểm tra file CSV cũ để xác định những ảnh đã chạy (Cơ chế Resume)
if os.path.exists(CSV_PATH):
    df_existing = pd.read_csv(CSV_PATH)
    processed_images = set(df_existing['image_path'].unique())
    print(f"Đã tìm thấy file CSV cũ. Bỏ qua {len(processed_images)} ảnh đã xử lý...")
else:
    processed_images = set()
    print("Tạo file CSV mới...")

# Lọc ra danh sách các ảnh CHƯA được xử lý
images_to_process = [item for item in saved_image_paths if item['rel_path'] not in processed_images]
print(f"Số lượng ảnh cần gọi API trong phiên này: {len(images_to_process)}")

# Vòng lặp với thanh tiến trình tqdm
for item in tqdm(images_to_process, desc="Đang sinh dữ liệu VQA"):
    rel_path = item['rel_path']
    abs_path = item['abs_path']
    label_name = item['label']

    # 1. Gọi API sinh Q&A
    qa_list = generate_vqa_for_image(abs_path, label_name)

    if qa_list:
        # 2. In ra màn hình để kiểm tra trực quan
        print(f"\n--- [THÀNH CÔNG] Ảnh: {rel_path} ---")
        print(json.dumps(qa_list, indent=2, ensure_ascii=False))

        # 3. Chuẩn bị dữ liệu để lưu ngay lập tức
        rows_to_save = []
        for qa in qa_list:
            rows_to_save.append({
                "image_path": rel_path,
                "question": qa.get("question", ""),
                "answer": qa.get("answer", "")
            })

        df_new = pd.DataFrame(rows_to_save)

        # 4. Ghi nối tiếp (Append) trực tiếp vào file CSV
        # Nếu file chưa tồn tại (ảnh đầu tiên), Pandas sẽ tự tạo Header. Các ảnh sau chỉ ghi nối thêm dòng.
        df_new.to_csv(CSV_PATH, mode='a', header=not os.path.exists(CSV_PATH), index=False, encoding='utf-8-sig')

    else:
        print(f"\n--- [LỖI] API không trả về kết quả cho ảnh: {rel_path} ---")

    # Nghỉ 4 giây tránh Rate Limit của Google (15 requests/phút)
    time.sleep(4)

print("\nHOÀN TẤT PHIÊN CHẠY! Hãy mở file CSV trên Drive để kiểm tra thành quả.")

Đã tìm thấy file CSV cũ. Bỏ qua 818 ảnh đã xử lý...
Số lượng ảnh cần gọi API trong phiên này: 1282


Đang sinh dữ liệu VQA:   0%|          | 0/1282 [00:00<?, ?it/s]


--- [THÀNH CÔNG] Ảnh: images/Banh duc/Banh duc_022.jpg ---
[
  {
    "question": "Món ăn trong hình là gì?",
    "answer": "Đây là món bánh đúc."
  },
  {
    "question": "Trên món ăn có rắc topping gì?",
    "answer": "Có đậu phộng rang giã nhỏ."
  },
  {
    "question": "Phần bánh có màu chủ đạo là gì?",
    "answer": "Bánh có màu xanh lá nhạt."
  }
]

--- [THÀNH CÔNG] Ảnh: images/Banh tet/Banh tet_049.jpg ---
[
  {
    "question": "Đây có phải là món bánh tét không?",
    "answer": "Phải, đây là bánh tét."
  },
  {
    "question": "Bánh tét trong ảnh có những màu sắc nào trên vỏ?",
    "answer": "Có màu xanh lá và tím."
  },
  {
    "question": "Bánh tét này đã được cắt thành lát chưa?",
    "answer": "Đã được cắt lát sẵn."
  }
]

--- [THÀNH CÔNG] Ảnh: images/Banh tet/Banh tet_050.jpg ---
[
  {
    "question": "Đây là món ăn gì?",
    "answer": "Bánh tét."
  },
  {
    "question": "Bánh được gói bằng gì?",
    "answer": "Lá chuối và lạt tre."
  },
  {
    "question": "Lá gói bánh c

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3289.18ms



--- [THÀNH CÔNG] Ảnh: images/Banh trang nuong/Banh trang nuong_013.jpg ---
[
  {
    "question": "Đây là món ăn gì?",
    "answer": "Bánh tráng nướng."
  },
  {
    "question": "Món này có trứng và xúc xích không?",
    "answer": "Có, trứng và xúc xích."
  },
  {
    "question": "Món ăn đang được nướng trên than phải không?",
    "answer": "Đúng, trên vỉ nướng."
  }
]

--- [THÀNH CÔNG] Ảnh: images/Banh trang nuong/Banh trang nuong_014.jpg ---
[
  {
    "question": "Đây là món ăn gì?",
    "answer": "Bánh tráng nướng Việt Nam."
  },
  {
    "question": "Các nguyên liệu chính là gì?",
    "answer": "Bánh tráng, trứng, xúc xích, sốt."
  },
  {
    "question": "Món ăn này đã được cắt chưa?",
    "answer": "Rồi, đã cắt thành nhiều miếng tam giác."
  }
]

--- [THÀNH CÔNG] Ảnh: images/Banh trang nuong/Banh trang nuong_015.jpg ---
[
  {
    "question": "Đây là món ăn gì?",
    "answer": "Banh trang nuong."
  },
  {
    "question": "Món ăn này có trứng và hành phi không?",
    "answer": "Có, c

...


--- [THÀNH CÔNG] Ảnh: images/Xoi xeo/Xoi xeo_060.jpg ---
[
  {
    "question": "Món ăn chính trong ảnh là gì?",
    "answer": "Xôi xéo."
  },
  {
    "question": "Món ăn này có đậu xanh không?",
    "answer": "Có, là phần đậu xanh nghiền."
  },
  {
    "question": "Xôi trong món ăn có màu gì?",
    "answer": "Màu vàng."
  }
]


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3086.10ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3770.04ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1975.12ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1421.03ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1417.52ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 6524.65ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2506.26ms



--- [THÀNH CÔNG] Ảnh: images/Xoi xeo/Xoi xeo_061.jpg ---
[
  {
    "question": "Đây có phải món xôi xeo không?",
    "answer": "Đúng, xôi xeo."
  },
  {
    "question": "Món ăn này gồm những nguyên liệu chính nào?",
    "answer": "Xôi, đậu xanh nghiền, hành phi."
  },
  {
    "question": "Món ăn được đặt trên bề mặt nào?",
    "answer": "Giấy báo."
  }
]


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 6274.25ms



--- [THÀNH CÔNG] Ảnh: images/Xoi xeo/Xoi xeo_062.jpg ---
[
  {
    "question": "Đây có phải món xôi xéo không?",
    "answer": "Đúng vậy."
  },
  {
    "question": "Món ăn này có thành phần đậu xanh không?",
    "answer": "Có, lớp vàng là đậu xanh."
  },
  {
    "question": "Lớp hành phi trên cùng có màu gì?",
    "answer": "Nâu vàng óng."
  }
]

--- [THÀNH CÔNG] Ảnh: images/Xoi xeo/Xoi xeo_063.jpg ---
[
  {
    "question": "Món ăn trong ảnh là gì?",
    "answer": "Xôi xéo."
  },
  {
    "question": "Món ăn này có hành phi không?",
    "answer": "Có, rất nhiều."
  },
  {
    "question": "Phần xôi có màu chủ đạo là gì?",
    "answer": "Màu vàng tươi."
  }
]

--- [THÀNH CÔNG] Ảnh: images/Xoi xeo/Xoi xeo_064.jpg ---
[
  {
    "question": "Đây có phải món xôi xéo không?",
    "answer": "Đúng vậy."
  },
  {
    "question": "Món ăn có những thành phần chính nào?",
    "answer": "Xôi nếp, đậu xanh, hành phi."
  },
  {
    "question": "Hành phi trên xôi có màu gì?",
    "answer": "Màu vàng nâ